# Evidence integration：分析上下文相关的表示扭曲

这是论文作者 notebook 的 Colab 运行副本。作者源码保存在 `../upstream/`，这里仅增加环境初始化并清除旧输出。

先运行下面的初始化单元，再选择 **Runtime → Restart session and run all**。

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/Heptazero/nn-labs.git"
REPO_DIR = Path("/content/nn-labs")

if not (REPO_DIR / ".git").exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
        check=True,
    )

SOURCE_DIR = REPO_DIR / "rnn-warping-2025" / "upstream"
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--editable",
        str(SOURCE_DIR),
    ],
    check=True,
)

RUN_DIR = SOURCE_DIR / "projects/dynamic_networks/evidence_integration"
os.chdir(RUN_DIR)
print(f"Environment ready. Working directory: {RUN_DIR}")


### Imports

In [ ]:
from jax import random
from jax import numpy as jnp

import jax

import equinox as eqx
from diffrax import *

from tqdm.auto import tqdm

import optax

import matplotlib.pyplot as plt
import matplotlib

import numpy as np

from functools import partial

from riemannian_dynamics.dynamic import rnn

import riemannian_dynamics as rd
from riemannian_dynamics.plotting import plotting2d, plotting3d, utils

utils.set_font(font_size=18)

#jax.config.update('jax_platforms', 'cpu')
#jax.config.update("jax_default_device", jax.devices('cpu')[0])

In [ ]:
cmap_e1 = utils.get_cmap_interpolated(*matplotlib.colormaps['Blues'](np.linspace(1, 0.1, 101)), *matplotlib.colormaps['Blues'](np.linspace(0.1, 1, 101)))
cmap_e2 = utils.get_cmap_interpolated(*matplotlib.colormaps['Greys'](np.linspace(1, 0.1, 101)), *matplotlib.colormaps['Greys'](np.linspace(0.1, 1, 101)))
cmap_traj = matplotlib.colormaps['YlGn']

### RNN definition

In [ ]:
key = random.PRNGKey(0)

hidden_dim = 100
input_dims = [1, 1, 2]
neuron_noise = 0.1 # fraction of noise
input_noise = 1.0 # fraction of noise
noise = 0.5
gain = 0.5
final_activation = lambda x: x

training_iterations = 3000
learning_rate = 5*10**-5
weight_decay = None

min_evidence= -0.2
max_evidence= 0.2
nb_evidence= 11

nb_noise = 3

max_time = 10.0

nb_time = 51

ts = jnp.linspace(0.0, max_time, nb_time)

model = rnn.RNN(jax.random.PRNGKey(0), hidden_dim, input_dims, 1, jax.nn.tanh, gain)

### Input definition

In [ ]:

# ===== evidences =====
evidences = jnp.linspace(min_evidence, max_evidence, nb_evidence).at[nb_evidence//2].set(0.0)
evidence_input = eqx.filter_vmap(LinearInterpolation, in_axes=(None, 0))(jnp.array([0.0, max_time]), jnp.stack([evidences, evidences], axis=-1)[..., None])

# ===== Context =====
contexts = jnp.stack(2*[jnp.eye(2)], axis=1)
context_input = eqx.filter_vmap(LinearInterpolation, in_axes=(None, 0))(jnp.linspace(0, max_time, 2), contexts)

# ===== Targets =====
target_ts = jnp.array([0.0, max_time])
target_points = jnp.concatenate([jnp.ones(1), jnp.ones(1)])

target_points = jax.vmap(jax.vmap(jax.vmap(jax.vmap(lambda c, x1, x2, y: (((x1>0)*1.0 - (x1<0)*1.0)*c[0] + ((x2>0)*1.0 - (x2<0)*1.0)*c[1])*y,
                                                    in_axes=(None, None, 0, None)),
                                                    in_axes=(None, 0, None, None)),
                                                    in_axes=(None, None, None, 0)),
                                                    in_axes=(0, None, None, None))(jnp.eye(2), evidences, evidences, target_points).transpose(0, 2, 3, 1)
target_output = eqx.filter_vmap(eqx.filter_vmap(eqx.filter_vmap(LinearInterpolation, in_axes=(None, 0)), in_axes=(None, 0)), in_axes=(None, 0))(target_ts, target_points)

### Save / Load model

We provided a trained model

In [ ]:
def save(filename, model):
    with open(filename, "wb") as f:
        eqx.tree_serialise_leaves(f, model)

def load(filename, model):
    with open(filename, "rb") as f:
        model = eqx.tree_deserialise_leaves(f, model)

    return model

model = load('./models/submission/model.eqx', model) ; training_iterations = 0
#save("./models/model.eqx", model)

### Solving the dynamical system

In [ ]:
@eqx.filter_jit
def solve(key, model, context_input, evidence, evidence2, noise):

    evidence_input = LinearInterpolation(jnp.array([0.0, max_time]), jnp.stack([evidence, evidence], axis=-1)[..., None])
    evidence_input2 = LinearInterpolation(jnp.array([0.0, max_time]), jnp.stack([evidence2, evidence2], axis=-1)[..., None])

    key, k1, k2, k3 = random.split(key, 4)
    bm = VirtualBrownianTree(shape=(hidden_dim,), t0=0.0, t1=max_time, tol=10**-2, key=k1)
    bm_input1 = VirtualBrownianTree(shape=(1,), t0=0.0, t1=max_time, tol=10**-2, key=k2)
    bm_input2 = VirtualBrownianTree(shape=(1,), t0=0.0, t1=max_time, tol=10**-2, key=k3)

    terms = [ODETerm(model.f()),
             ODETerm(lambda t, x, args: model.g_i(0)(t, x, args) @ evidence_input.evaluate(t)),
             ODETerm(lambda t, x, args: model.g_i(1)(t, x, args) @ evidence_input2.evaluate(t)),
             ODETerm(lambda t, x, args: model.g_i(2)(t, x, args) @ context_input.evaluate(t)),
             ]

    if noise != 0:
        terms = terms + [ControlTerm(model.h_sigma(noise*neuron_noise), bm),
                         ControlTerm(model.h_g_i(0, noise*input_noise), bm_input1),
                         ControlTerm(model.h_g_i(1, noise*input_noise), bm_input2)]

    terms = MultiTerm(*terms)

    sol = diffeqsolve(terms, Heun(), 0.0, max_time, 0.01, model.x0(),
                       saveat=SaveAt(ts=ts),
                       stepsize_controller=PIDController(rtol=10**-2, atol=10**-3)
                      )

    return sol.ys

In [ ]:
solve_vmap = eqx.filter_vmap(eqx.filter_vmap(eqx.filter_vmap(eqx.filter_vmap(solve,
                                                                             in_axes=(None, None, None, None, 0, None)),
                                                                            in_axes=(None, None, None, 0, None, None)),
                                                                            in_axes=(None, None, 0, None, None, None)),
                                                                            in_axes=(0, None, None, None, None, None))

In [ ]:
def evaluate_(path, ts):

    return eqx.filter_vmap(path.evaluate)(ts)

In [ ]:
ys = eqx.filter_vmap(eqx.filter_vmap(eqx.filter_vmap(evaluate_, in_axes=(0, None)), in_axes=(0, None)), in_axes=(0, None))(target_output, ts)
ds = eqx.filter_vmap(evaluate_, in_axes=(0, None))(evidence_input, ts)
cs = eqx.filter_vmap(evaluate_, in_axes=(0, None))(context_input, ts)

### Training

In [ ]:
def loss_(model, xs):

    ys_hat = final_activation(model.d(xs))

    dys = (ys[:, :, :, :, None] - ys_hat)[..., -1, :]

    return jnp.mean(dys**2)


@eqx.filter_jit
def loss(model, key):

    keys = random.split(key, nb_noise)

    xs = solve_vmap(keys, model, context_input, evidences, evidences, noise)

    return loss_(model, xs)

@eqx.filter_jit
def loss_test(model):

    xs = solve_vmap(random.PRNGKey(0)[None], model, context_input, evidences, evidences, 0.0)

    return loss_(model, xs)

@eqx.filter_jit
def step(key, model, optim, opt_state):

    l_test = loss_test(model)

    l_train, grads = eqx.filter_value_and_grad(loss)(model, key)

    updates, opt_state = optim.update(grads, opt_state, model)

    model = eqx.apply_updates(model, updates)

    return model, l_train, l_test, opt_state

In [ ]:
iterator = tqdm(range(training_iterations))

optim = optax.adamw(learning_rate=learning_rate, weight_decay=weight_decay) if weight_decay is not None else optax.adam(learning_rate=learning_rate)

opt_state = optim.init(eqx.filter(model, eqx.is_array))

losses = []

for iteration in iterator:

    key, subkey = random.split(key)
    model, l_train, l_test, opt_state = step(subkey, model, optim, opt_state)

    losses.append([l_train, l_test])

    iterator.set_description(f"Train loss: {l_train:.6f}, Test loss: {l_test:.6f}")

losses = jnp.array(losses)

### Some plotting

We evaluate x with the trained model

In [ ]:
evidence_id = nb_evidence//2
cue_id = 0

xs = solve_vmap(random.PRNGKey(0)[None], model, context_input, evidences, evidences, 0.0)[0]
ys_hat = final_activation(model.d(xs))

print(xs.shape, ys_hat.shape, ys.shape)

In [ ]:
axs, _, fig = rd.plotting.utils.get_ax_gridspec(1, 10, cols_3d=(0, 1, 9), dpi=80)

for ax_ in axs.T:

    # ===== 3D plot =====

    xs_centered = xs - xs.mean(axis=(0, 1, 2, 3))

    U, S, V = jnp.linalg.svd(xs_centered[:, :, :].reshape(-1, hidden_dim), full_matrices=False) # Center ?

    #xs_pca = xs[..., 1:4] #@ V[:3].T
    V_ = np.zeros((3, hidden_dim))
    for j in range(3):
        V_[j, np.argmax(np.abs(V[j]))] = 1.0

    xs_pca = xs_centered @ V_[:3].T

    xs_pca_time = xs_pca.at[..., 0].set(ts[None, None, None, :])

    sample_ts = [1, 12, 24, 36, -1]

    for x, a, cmap, transpose in zip(xs_pca_time, ax_[:2], [cmap_e1, cmap_e2], [(1, 0, 2), (0, 1, 2)]):
        for i in sample_ts:
            plotting3d.plot_manifold(a, x[:, :, i].transpose(*transpose), cmap, cla=False, gridlines=False)

            a.scatter(*x[nb_evidence - 4, nb_evidence - 4, i], color=cmap_traj(ts[i]/np.max(ts)), s=20, linewidth=0.0, alpha=1.0)
        plotting3d.plot_with_gradient_3d(a, *x[nb_evidence - 4, nb_evidence - 4, :].T,
                                             cmap=cmap_traj,
                                             gradient=np.linspace(0.1, 1.0, len(ts)),
                                             linewidth=1.0, alpha=1.0, set_lim=False)

        a.set_aspect('equal')
        a.set_xlabel('Time'), a.set_ylabel('Neuron 1'), a.set_zlabel('Neuron 2')
        a.view_init(25, -30, 0)
        a.set_box_aspect((3, (jnp.max(x[..., 1])-jnp.min(x[..., 1]))/(jnp.max(x[..., 2])-jnp.min(x[..., 2])), 1))

        utils.set_pannels_3d(a)
        utils.remove_ticks(a)

    # ===== Time series =====
    ax = ax_[2]
    colors = cmap_e2((evidences-jnp.min(evidences))/(jnp.max(evidences)-jnp.min(evidences)))
    for i in range(nb_evidence):
        ax.plot(ts, xs[cue_id, i, evidence_id, :, 1], color=colors[i])

    utils.set_bottom_left_axis(ax)
    ax.set_xlabel('Time'), ax.set_ylabel('RNN activity')

    # ===== evidence input =====
    ax = ax_[3]

    axs_ = utils.add_vertical_axes(ax, 2, spacing=0.2)

    ax = axs_[0]
    colors = cmap_e1((evidences-jnp.min(evidences))/(jnp.max(evidences)-jnp.min(evidences)))
    for i in range(nb_evidence):
        ax.plot(ts, ds[i], color=colors[i])

    ax.set_ylim(evidences[0]-0.05, evidences[-1]+0.05)

    utils.set_bottom_left_axis(ax)
    ax.set_xlabel('Time'), ax.set_ylabel('Input')

    # ===== Go input =====
    ax = axs_[1]
    colors = cmap_e2((evidences-jnp.min(evidences))/(jnp.max(evidences)-jnp.min(evidences)))
    for i in range(nb_evidence):
        ax.plot(ts, ds[i], color=colors[i])

    ax.set_ylim(evidences[0]-0.05, evidences[-1]+0.05)

    utils.set_bottom_left_axis(ax)
    ax.set_xlabel('Time'), ax.set_ylabel('Input')

    # ===== Target output =====
    ax = ax_[4]

    axs_ = utils.add_vertical_axes(ax, 2, spacing=0.2)

    ax = axs_[0]
    for i in range(nb_evidence):
        ax.plot(ts, ys[cue_id, i, evidence_id], color=colors[i])

    ax.set_ylim(-1.05, 1.15)

    utils.set_bottom_left_axis(ax)
    ax.set_xlabel('Time'), ax.set_ylabel('Target')

    # ===== RNN output =====
    ax = axs_[1]
    for i in range(nb_evidence):
        ax.plot(ts, ys_hat[cue_id, i, evidence_id], color=colors[i])

    ax.set_ylim(-1.05, 1.15)

    utils.set_bottom_left_axis(ax)
    ax.set_xlabel('Time'), ax.set_ylabel('Output')

    # ===== Eig =====
    ax = ax_[5]

    L, V = np.linalg.eig(model.W)

    ax.scatter(L.real, L.imag)

    utils.set_centered_axes(ax)
    rmax = jnp.max(jnp.abs(L))
    ax.set_xlim(-rmax, rmax), ax.set_ylim(-rmax, rmax)

    # ===== Couple of neurons =====
    ax = ax_[6]

    colors = cmap_e1((evidences-jnp.min(evidences))/(jnp.max(evidences)-jnp.min(evidences)))
    for i in range(20):
        ax.plot(ts, xs[cue_id, 0, evidence_id, :, i])

    utils.set_bottom_left_axis(ax)
    ax.set_xlabel('Time'), ax.set_ylabel('RNN activity')

    # ===== S.V. =====
    ax = ax_[7]

    S = jnp.linalg.svd(xs.reshape(-1, xs.shape[-1]), full_matrices=False, compute_uv=False)

    ax.plot(jnp.arange(20)+1, S[:20], '-o')

    #ax.set_yscale('log')

    utils.set_bottom_left_axis(ax)
    ax.set_xlabel('Index'), ax.set_ylabel('S.V.')


    # ===== Loss =====
    ax = ax_[8]

    if len(losses) != 0:
        ax.plot(losses[:, 1])

    utils.set_bottom_left_axis(ax)
    ax.set_xlabel('Iteration'), ax.set_ylabel('Test loss')


    # ===== 3D plot =====
    ax = ax_[9]

    x = xs_pca[:, :, :, :15]#[:, evidence_id:evidence_id+2, evidence_id:evidence_id+2, 20:22]
    t = ts[:15]#[20:22]

    ts_normalized = t/jnp.max(t)
    ts_normalized = np.broadcast_to(ts_normalized[None, None, None, :, None], x.shape)
    colors = matplotlib.colormaps['pink_r'](ts_normalized).reshape(-1, 3, 4)[:, 0, :]
    #colors[..., 3] = 1.0

    print(colors.shape, xs_pca.shape)
    cmap = matplotlib.colormaps['Greys']

    #ax.scatter(*x.reshape(-1, x.shape[-1]).T, c=colors, s=2, linewidth=0.0, alpha=0.2)
    for k, cmap in enumerate([cmap_e1, cmap_e2]):
        for i in range(nb_evidence):
            for j in range(nb_evidence):
                #ax.plot(*x[k, i, j].T, color=cmap((i+1)/nb_evidence), alpha=0.5)
                plotting3d.plot_with_gradient_3d(ax, *x[k, i, j].T,
                                                 cmap=cmap_traj, gradient=np.linspace(0.1, 1.0, len(t)), linewidth=1.0, alpha=0.99)

        ax.plot_surface(*x[k, :, :, -1].T, shade=False, color=matplotlib.colormaps['Greys'](1.0)[:3], linewidth=0.0, alpha=0.3)

    ax.view_init(30, -60, 0)

    utils.set_bottom_left_axis(ax)
    ax.set_xlabel('Neuron 1'), ax.set_ylabel('Neuron 2'), ax.set_ylabel('Neuron 3')

plt.savefig('./plots/evidence_integration_main.pdf')

plt.show()

### Metric

In [ ]:
solve_vmap_evidence = eqx.filter_vmap(eqx.filter_vmap(solve, in_axes=(None, None, 0, None, None, None)), in_axes=(0, None, None, None, None, None))

solve_vmap_evidence_ = lambda e1, e2: solve_vmap_evidence(random.PRNGKey(0)[None], model, context_input, e1, e2, 0.0)[0]
dx_devidence_1 = jnp.stack([jax.vmap(jax.jacrev(solve_vmap_evidence_, argnums=(0)), in_axes=(None, 0), out_axes=1)(ei, evidences) for ei in evidences], axis=1)
dx_devidence_2 = jnp.stack([jax.vmap(jax.jacrev(solve_vmap_evidence_, argnums=(1)), in_axes=(0, None), out_axes=1)(evidences, ei) for ei in evidences], axis=2)

e1 = ((model.g_i(0)(None, None, None) @ evidences[None, :]).T)[None, :, None, None, :]
e2 = ((model.g_i(1)(None, None, None) @ evidences[None, :]).T)[None, None, :, None, :]
c = (cs @ model.g_i(2)(None, None, None).T)[:, None, None]
f = jax.vmap(jax.vmap(jax.vmap(jax.vmap(model.F))))(xs)

dx_dt = f + e1 + e2 + c

basis = jnp.stack([dx_dt, dx_devidence_1, dx_devidence_2], axis=-2)

#basis = basis / (jnp.linalg.norm(basis, axis=-1, keepdims=True) + 10**-8)

metric = jnp.einsum('...ij,...kj->...ik', basis, basis)

print(f'basis: {basis.shape}  |  metric: {metric.shape}')

Metric over a different combination of levels of evidences, across two contexts

In [ ]:
nb_evidence_sample = 11

axs, _, fig1 = rd.plotting.utils.get_ax_gridspec(nb_evidence_sample, nb_evidence_sample, cols_3d=(), dpi=50)
axs1, _, fig2 = rd.plotting.utils.get_ax_gridspec(nb_evidence_sample, nb_evidence_sample, cols_3d=(), dpi=50)
axs = np.stack([axs, axs1], axis=0)

time_id = 3

vmax = jnp.quantile(jnp.abs(metric[:, :, :, time_id]), 1.0)

for k, axs_c in enumerate(axs):
    vmax = jnp.quantile(jnp.abs(metric[k, :, :, time_id]), 1.0)

    for j, ax_ in enumerate(axs_c):
        for i, ax in enumerate(ax_):
            m = metric[k, i*(nb_evidence//(nb_evidence_sample-1)), j*(nb_evidence//(nb_evidence_sample-1)), time_id]
            #vmax = jnp.max(jnp.abs(m))
            m = m.at[jnp.triu_indices_from(m, k=1)].set(jnp.nan)
            im = ax.imshow(m, vmin=-vmax, vmax=vmax, cmap='RdBu_r')

            utils.remove_axes(ax)
            utils.remove_ticks(ax)
            if i == 0: ax.set_title(f'E = {jnp.round(evidences[j*(nb_evidence//(nb_evidence_sample-1))], 4):.3f}')
            if j == 0: ax.set_ylabel(f'E = {jnp.round(evidences[i*(nb_evidence//(nb_evidence_sample-1))], 4):.3f}')

    plt.colorbar(im, ax=ax, shrink=0.5)

fig1.savefig(f'./plots/evidence_integration_metric_t{time_id}_c1.pdf')
fig2.savefig(f'./plots/evidence_integration_metric_t{time_id}_c2.pdf')
plt.show()

### Eigenvalues of the metric

Averaged over the relevant evidence (error bars irrelevant evidence)

In [ ]:
axs, _, fig = rd.plotting.utils.get_ax_gridspec(2, nb_evidence_sample, cols_3d=(), dpi=50)

S = jax.vmap(jax.vmap(jax.vmap(jax.vmap(partial(jnp.linalg.svd, compute_uv=False)))))(metric)

S = S**2

cmap = matplotlib.colormaps['Set2']

for i, ax_ in enumerate(axs):
    for j, ax in enumerate(ax_):
        s = S[j, i*(nb_evidence//(nb_evidence_sample-1)) if j==0 else slice(None), i*(nb_evidence//(nb_evidence_sample-1)) if j==1 else slice(None)]

        #s = s / jnp.max(s, axis=-1, keepdims=True)

        s_med = jnp.median(s, axis=0)
        s_min = jnp.quantile(s, axis=0, q=0.05)
        s_max = jnp.quantile(s, axis=0, q=0.95)

        s_min, s_med, s_max = s_min/jnp.max(s_med, axis=1, keepdims=True), s_med/jnp.max(s_med, axis=1, keepdims=True), s_max/jnp.max(s_med, axis=1, keepdims=True)

        for k in range(s_min.shape[-1]):
            ax.fill_between(ts, s_min[:, k], s_max[:, k], alpha=0.2, color=cmap(k))
            ax.plot(ts, s_med[:, k], color=cmap(k))

        ax.set_xlabel('Time')
        ax.set_ylabel('S.V. metric')

        utils.set_bottom_left_axis(ax)

utils.set_shared_lims(*axs.reshape(-1))
plt.savefig('./plots/rank_metric.pdf')
plt.show()

Pushforward of the decoder applied to the basis of the tangent space

In [ ]:
pushforward_basis_decoder = jax.vmap(jax.vmap(jax.vmap(jax.vmap(model.d))))(basis) / (jnp.linalg.norm(model.D, axis=-1)*jnp.linalg.norm(basis, axis=-1, keepdims=True) + 10**-8)
pushforward_decoder = jax.vmap(jax.vmap(jax.vmap(jax.vmap(jax.jacrev(model.d)))))(xs)

pushforward_decoder_norm = pushforward_decoder / (jnp.linalg.norm(pushforward_decoder, axis=-1, keepdims=True) + 10**-8)
basis_norm = basis / (jnp.linalg.norm(basis, axis=-1, keepdims=True) + 10**-8)
pushforward_basis_decoder_norm = jnp.einsum('...ij,...kj->...ki', pushforward_decoder_norm, basis_norm)

### Aligment of decoder

In [ ]:
axs, _, fig = rd.plotting.utils.get_ax_gridspec(1, 2, cols_3d=(), dpi=50)

med_pbd = jnp.median(pushforward_basis_decoder_norm[:, :, :, :, :, 0], axis=(1, 2))
min_pbd = jnp.quantile(pushforward_basis_decoder_norm[:, :, :, :, :, 0], 0.1,  axis=(1, 2))
max_pbd = jnp.quantile(pushforward_basis_decoder_norm[:, :, :, :, :, 0], 0.9, axis=(1, 2))

cmap = matplotlib.colormaps['Set1']

for i, ax in enumerate(axs[:, 0]):

    for k, lb in enumerate(['t', 'e1', 'e2']):
        if  k != 0:
            ax.plot(ts, med_pbd[i, :, k], label=lb, color=cmap(k))
            ax.fill_between(ts, min_pbd[i, :, k], max_pbd[i, :, k], alpha=0.2, color=cmap(k))
    ax.axhline(0, linestyle='--', color='black')
    #ax.set_ylim(jnp.min(pushforward_basis_decoder_norm[l, :, :, :, 1:]), jnp.max(pushforward_basis_decoder_norm[l, :, :, :, 1:]))

    utils.set_bottom_left_axis(ax)

    ax.set_xlabel('Time')
    ax.set_ylabel('Decoder alignment')
    ax.set_title(f'Context {i}\n')
    ax.legend()

utils.set_shared_lims(*axs[:,0])

plt.savefig('./plots/de_decoder_alignment.pdf')
plt.show()

### Distribution of responses

To small changes along basis vectors

In [ ]:
from matplotlib.patches import Ellipse

def plot_gaussian_ellipse(ax, x, y, nstd=2, **kwargs):
    """
    Fit a Gaussian to (x, y) points and plot its n-sigma error ellipse on the given Axes.

    Args:
        ax : matplotlib Axes
        x : (N,) array of x-coordinates
        y : (N,) array of y-coordinates
        nstd : Number of standard deviations for the ellipse (default 2)
        kwargs : Additional arguments passed to Ellipse (e.g., edgecolor, facecolor)
    """
    # Stack into (N,2) array
    X = np.column_stack((x, y))

    # Compute mean and covariance
    mu = np.mean(X, axis=0)
    cov = np.cov(X, rowvar=False)

    # Eigenvalues and eigenvectors
    vals, vecs = np.linalg.eigh(cov)
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]

    # Compute angle
    theta = np.degrees(np.arctan2(*vecs[:,0][::-1]))

    # Width and height
    width, height = 2 * nstd * np.sqrt(vals)
    ellipse = Ellipse(xy=mu, width=width, height=height, angle=theta, **kwargs)

    ax.add_patch(ellipse)
    return ellipse

In [ ]:
axs, _, fig = rd.plotting.utils.get_ax_gridspec(2, 2, cols_3d=(), dpi=50)

time_id = -1

cmap = matplotlib.colormaps['Set2']

print(basis_norm.shape)
#basis_norm_proj_time = jnp.einsum('...ki,...i->...k', basis_norm, basis_norm[..., 0, :])
b = basis_norm

for i, (ax, title) in enumerate(zip(axs[:, :], ['Same context', 'Opposite context'])):

    plot_gaussian_ellipse(ax[0], b[i, evidence_id, evidence_id, time_id, 0], b[i, evidence_id, evidence_id, time_id, 1], nstd=2,
                          facecolor=list(cmap(0))[:3]+[0.1], edgecolor=list(cmap(0))[:3]+[1.0], linewidth=3.0)
    plot_gaussian_ellipse(ax[1], b[1-i, evidence_id, evidence_id, time_id, 0], b[1-i, evidence_id, evidence_id, time_id, 2], nstd=2,
                          facecolor=list(cmap(1))[:3]+[0.1], edgecolor=list(cmap(1))[:3]+[1.0], linewidth=3.0)
    #ax.set_aspect('equal')

    ax[0].scatter(b[i, evidence_id, evidence_id, time_id, 0], b[i, evidence_id, evidence_id, time_id, 1], color=cmap(0))
    ax[1].scatter(b[1-i, evidence_id, evidence_id, time_id, 0], b[1-i, evidence_id, evidence_id, time_id, 2], color=cmap(1))

    utils.remove_axes(ax[0])
    utils.remove_axes(ax[1])
    ax[0].set_title(title)

utils.set_shared_lims(*axs.reshape(-1))

plt.savefig('./plots/de_dt_alignment.pdf')
plt.show()

### Singular values of the weights

Or rather the change in weights

In [ ]:
# This assumes that the model uses the 0 key
model_init = rnn.RNN(jax.random.PRNGKey(0), hidden_dim, input_dims, 1, jax.nn.tanh, gain)

U, S, V = jnp.linalg.svd(model.W - model_init.W)

In [ ]:
axs, _, fig = rd.plotting.utils.get_ax_gridspec(1, 1, cols_3d=(), dpi=50)

axs[0, 0].plot(S, '-o')
plt.savefig('./plots/svd_dw.pdf')

### Alignment of basis vectors to singular vectors

In [ ]:
def ivmap(fn, n):
    for _ in range(n):
        fn = jax.vmap(fn)
    return fn

In [ ]:
basis_on_rsv = basis @ V.T

In [ ]:
nb_ts_sample = 11

In [ ]:
axs, _, fig = rd.plotting.utils.get_ax_gridspec(nb_ts_sample, 2, cols_3d=(), dpi=50)

for j, ax_ in enumerate(axs.T):
    for i, (ax, title) in enumerate(zip(ax_, ['Same context', 'Opposite context'])):
        b = basis_on_rsv[i, evidence_id, evidence_id, j*len(ts)//nb_ts_sample, :, :4]
        im = ax.imshow(b, cmap='RdBu_r', vmin=-jnp.max(jnp.abs(b)), vmax=jnp.max(jnp.abs(b)))

        if j == len(axs.T)-1: ax.set_xlabel('Right singular vector')
        if i == 0: ax.set_ylabel('Basis')

        #plt.colorbar(im, ax=ax, shrink=0.25)

plt.savefig('./plots/sv_alignment.pdf')
plt.show()

In [ ]:
axs, _, fig = rd.plotting.utils.get_ax_gridspec(nb_ts_sample, 2, cols_3d=(), dpi=50)

labels = ['e1', 'e2']  # Labels for the bars

cmap = matplotlib.colormaps['Set2']

for j, ax_ in enumerate(axs.T):
    for i, (ax, title) in enumerate(zip(ax_, ['Same context', 'Opposite context'])):
        b = basis_on_rsv[:, evidence_id, evidence_id, j*len(ts)//nb_ts_sample, :, :4]
        b_std = jnp.std(basis_on_rsv[:, :, :, j*len(ts)//nb_ts_sample], axis=(1, 2))

        # Plot horizontal bars
        ax.barh(jnp.arange(2), b[i, ..., 1:3, 0], xerr=b_std[i, ..., 1:3, 0], color=cmap(0), height=0.2, label='Condition 1', capsize=5)
        ax.barh(jnp.arange(2) + 0.25, b[1-i, ..., 1:3, 0], xerr=b_std[1-i, ..., 1:3, 0], color=cmap(1), height=0.2, label='Condition 2', capsize=5)

        #add err

        if j == len(axs.T) - 1:
            ax.set_xlabel('Basis')  # Horizontal axis now shows basis
        if i == 0:
            ax.set_ylabel('Right singular vector')  # Vertical axis shows "rsv"

        #ax.set_xlim(-jnp.max(jnp.abs(b[..., 0])), jnp.max(jnp.abs(b[..., 0])))

        # Add A, B, C as yticks
        ax.set_yticks(jnp.arange(2) + 0.125)
        ax.set_yticklabels(labels)

        utils.remove_axes(ax)  # Keep bottom and left if you want to see labels

plt.savefig('./plots/top_sv_alignment.pdf')
plt.show()

### Plotting the gridlines

In [ ]:
def normalize(ts, min_t, max_t):

    ts = (ts - np.min(ts))
    ts = ts/(np.max(ts) - np.min(ts))
    ts = ts * (max_t - min_t)
    ts = ts + min_t

    return ts

def plot_manifold(ax, xs, cmap, min_alpha=0.7, max_alpha=0.7, zorder=2, cla=True):

    if cla: ax.cla()

    #min_alpha, max_alpha = 0.0, 0.5
    alpha_decay = 1 - np.exp(np.linspace(-5, 0, xs.shape[0]))
    alpha_decay = normalize(alpha_decay, min_alpha, max_alpha)

    gradient_ts = np.concatenate([np.linspace(0, 1, xs.shape[1]-1), np.array([1.0])])
    colors = cmap(np.tile(gradient_ts, (xs.shape[0], 1)))
    colors[..., 3] = np.tile(alpha_decay, (xs.shape[1], 1)).T
    colors[..., 3] = min_alpha

    ax.plot_surface(xs[..., 0], xs[..., 1], xs[..., 2],
                    facecolors=colors,
                    linewidth=0.02,
                    ccount=xs.shape[0], rcount=xs.shape[1],
                    shade=False,
                    zorder=zorder)


In [ ]:
axs, _, fig = rd.plotting.utils.get_ax_gridspec(2, 2, cols_3d=(0, 1), dpi=80)

integrated_metric_e1 = jnp.cumsum(metric[..., 1, 1]**0.5, axis=1)
integrated_metric_e2 = jnp.cumsum(metric[..., 2, 2]**0.5, axis=2)

integrated_metric = jnp.stack([integrated_metric_e1, integrated_metric_e2], axis=-1)

view_init1 = (75, -120, -120)
view_init2 = (90, 0, 0)

# =========== t_max ==============
time_id = -1

m = jnp.stack([integrated_metric[..., 0], integrated_metric[..., 1], jnp.zeros_like(integrated_metric[..., 1])], axis=-1)
m = np.array(m)

m1 = (m[0, :, :, time_id] - m[0, nb_evidence//2, nb_evidence//2, time_id]).transpose(1, 0, 2)
m2 = m[1, :, :, time_id] - m[1, nb_evidence//2, nb_evidence//2, time_id]

ax = axs[1, 0]

plot_manifold(ax, m1, cmap=cmap_e1)
plot_manifold(ax, m2, cmap=cmap_e2, cla=False)

ax.set_title(f'T={ts[-1]}')
ax.set_aspect('equal')
ax.view_init(90, 0)
#ax.axis('off')
ax.view_init(*view_init2)

ax = axs[1, 1]

m2[..., 2] += 20.0

plot_manifold(ax, m1, cmap=cmap_e1)
plot_manifold(ax, m2, cmap=cmap_e2, cla=False)

ax.set_title(f'T={ts[-1]}')
ax.set_aspect('equal')
ax.view_init(90, 0)
#ax.axis('off')
ax.view_init(*view_init1)


# =========== t_min ==============

time_id = 1

m1 = (m[0, :, :, time_id] - m[0, nb_evidence//2, nb_evidence//2, time_id]).transpose(1, 0, 2)
m2 = m[1, :, :, time_id] - m[1, nb_evidence//2, nb_evidence//2, time_id]

ax = axs[0, 0]

plot_manifold(ax, m1, cmap=cmap_e1)
plot_manifold(ax, m2, cmap=cmap_e2, cla=False)

ax.set_aspect('equal')
ax.view_init(*view_init2)
#ax.axis('off')
ax.set_title(f'T={ts[time_id]:.3f}')

ax = axs[0, 1]

m2[..., 2] += 10.0

plot_manifold(ax, m1, cmap=cmap_e1)
plot_manifold(ax, m2, cmap=cmap_e2, cla=False)

ax.set_aspect('equal')
ax.view_init(*view_init1)
#ax.axis('off')
ax.set_title(f'T={ts[time_id]:.3f}')

plt.savefig('./plots/metric_e1_e2.pdf')
plt.show()


In [ ]:
axs, _, fig = rd.plotting.utils.get_ax_gridspec(2, 2, cols_3d=(0, 1, 2, 3), dpi=80)

integrated_metric_e1 = jnp.cumsum(metric[..., 1, 1], axis=1)
integrated_metric_e2 = jnp.cumsum(metric[..., 2, 2], axis=2)

integrated_metric = jnp.stack([integrated_metric_e1, integrated_metric_e2], axis=-1)

view_init1 = (75, -120, -120)
view_init2 = (90, 0, 0)

# =========== t_max ==============
time_id = -1

m = jnp.stack([integrated_metric[..., 0], integrated_metric[..., 1], jnp.zeros_like(integrated_metric[..., 1])], axis=-1)
m = np.array(m)

m[:, :, :, :, :1] -= (m[:, nb_evidence//2:nb_evidence//2+1, :, :, :1] + 10**-8)
m[:, :, :, :, 1:2] -= (m[:, :, nb_evidence//2:nb_evidence//2+1, :, 1:2] + 10**-8)

m[:, :nb_evidence//2, :, :, :1] /= (m[:, :1, :, :, :1] + 10**-8)/evidences[0]
m[:, :, :nb_evidence//2, :, 1:2] /= (m[:, :, :1, :, 1:2] + 10**-8)/evidences[0]

m[:, nb_evidence//2:, :, :, :1] /= (m[:, -1:, :, :, :1] + 10**-8)/evidences[-1]
m[:, :, nb_evidence//2:, :, 1:2] /= (m[:, :, -1:, :, 1:2] + 10**-8)/evidences[-1]

m1 = m[0, :, :, time_id].transpose(1, 0, 2)
m2 = m[1, :, :, time_id]

plot_manifold(axs[0, 0], m1, cmap=cmap_e1, min_alpha=0.9)
plot_manifold(axs[0, 1], m2, cmap=cmap_e2, min_alpha=0.9)

axs[0, 0].set_title(f'T={ts[time_id]:.3f}')

# =========== t_min ==============

time_id = 1

m1 = m[0, :, :, time_id].transpose(1, 0, 2)
m2 = m[1, :, :, time_id]

plot_manifold(axs[1, 0], m1, cmap=cmap_e1, min_alpha=0.9)
plot_manifold(axs[1, 1], m2, cmap=cmap_e2, cla=False, min_alpha=0.9)

axs[1, 0].set_title(f'T={ts[time_id]:.3f}')

for ax_ in axs:
    for ax in ax_:
        ax.set_aspect('equal')
        ax.view_init(90, 0)
        ax.axis('off')

plt.savefig('./plots/metric_e1_e2_pullback.pdf')
plt.show()

### Example outputs

In [ ]:
xs_noisy = solve_vmap(random.PRNGKey(3)[None], model, context_input, evidences, evidences, noise)[0]
ys_hat_noisy = final_activation(model.d(xs_noisy))

In [ ]:
key_, k1_, k2_, k3_ = random.split(key, 4)
bm = VirtualBrownianTree(shape=(hidden_dim,), t0=0.0, t1=max_time, tol=10**-2, key=k1_)
bm_input1 = VirtualBrownianTree(shape=(1,), t0=0.0, t1=max_time, tol=10**-2, key=k2_)
bm_input2 = VirtualBrownianTree(shape=(1,), t0=0.0, t1=max_time, tol=10**-2, key=k3_)

noise_realisation = evaluate_(bm_input1, ts)

ds_noisy = ds+noise*noise_realisation

In [ ]:
axs, _, fig = rd.plotting.utils.get_ax_gridspec(1, 1, cols_3d=(), dpi=80)

axs = utils.add_vertical_axes(axs[0, 0], 2)

# ====== Inputs ======
ax = axs[0]
for i in range(nb_evidence):
    ax.plot(ts, ys_hat_noisy[0, nb_evidence-1, i, :, 0],color=cmap_e2(i/(nb_evidence-1)))

utils.set_bottom_left_axis(ax)

ax.set_ylim(-1, 1)
ax.set_xlabel('Time')
ax.set_ylabel('Output')

# ====== Output ======
ax = axs[1]
for i in range(nb_evidence):
    ax.plot(ts, ys_hat_noisy[0, i, nb_evidence//2, :, 0],color=cmap_e1(i/(nb_evidence-1)))

utils.set_bottom_left_axis(ax)
ax.set_ylim(-1, 1)

ax.set_xlabel('Time')
ax.set_ylabel('Output')

plt.savefig('./plots/noisy_output.pdf')

plt.show()